# xy — demo

Millions of points, interactive, in a notebook. Pan by dragging, zoom with the wheel, double-click to reset.
Shift-drag a scatter to box-select. One declarative API over one engine: the Reflex-flavored `xy.scatter_chart(...)` composition.

In [ ]:
import numpy as np

import xy
import xy.kernels

rng = np.random.default_rng(0)
print("kernel backend:", xy.kernels.BACKEND)

## 10M-point line (ships decimated: ≤4 points per pixel column)

In [ ]:
n = 10_000_000
t = np.datetime64("2015-01-01") + np.arange(n).astype("timedelta64[s]")
y = np.cumsum(rng.normal(size=n))
y[n // 2] += 80  # a single-sample spike M4 must preserve

In [ ]:
xy.line_chart(
    xy.line(x=t, y=y, name="random walk"),
    xy.x_axis(label="time"),
    xy.y_axis(label="value"),
    title="10M points — wheel to zoom into the spike",
    width=950,
    height=420,
)

## Composition API (Reflex-flavored) + per-point color/size + events
Hover a point for its exact f64 row; shift-drag to box-select (unselected marks dim).

In [ ]:
m = 20_000
xs = rng.normal(0, 1, m)
ys = xs * 0.5 + rng.normal(0, 0.6, m)
depth = xs**2 + ys**2  # continuous -> viridis colormap
weight = np.abs(rng.normal(size=m))  # continuous -> size 3..18 px

xy.scatter_chart(
    xy.scatter(
        x=xs, y=ys, color=depth, size=weight, colormap="viridis", size_range=(3, 18), opacity=0.7
    ),
    xy.x_axis(label="x"),
    xy.y_axis(label="y"),
    title="colored + sized scatter",
    width=950,
    height=420,
    on_select=lambda sel: print(len(sel), "points selected"),
)

## data= + column names (the Reflex data_key idiom), categorical palette

In [ ]:
df = {
    "x": xs,
    "y": ys,
    "grp": np.array(["alpha", "beta", "gamma"])[rng.integers(0, 3, m)],
}
xy.scatter_chart(
    xy.scatter(x="x", y="y", color="grp", data=df, size=4.0, opacity=0.6),
    title="categorical scatter (color='grp')",
    width=950,
    height=420,
)

## 30M-point scatter -> Tier-2 density surface (§5)
Above ~200k points scatter auto-aggregates: the kernel bins the viewport into a
count grid, the client colormaps it on the GPU, and zoom re-bins the visible window.

In [ ]:
big = 30_000_000
bx = rng.normal(0, 1, big)
by = bx * 0.7 + rng.normal(0, 0.5, big)

chart = xy.scatter_chart(
    xy.scatter(
        x=bx,
        y=by,
    ),
    title="30M points (density)",
    width=950,
    height=420,
)
print("tier:", chart.figure().build_payload()[0]["traces"][0]["tier"])
chart

## The memory story (§27: if it isn't in the report, it isn't real)

In [ ]:
chart = xy.line_chart(xy.line(t, y))
report = chart.memory_report()
print(f"canonical: {report['canonical_bytes'] / 2**20:.1f} MB kernel-side")
print(f"first-paint transport: {report['transport_bytes_first_paint'] / 2**10:.1f} KB")
print(f"transport bytes/point: {report['transport_bytes_per_point']:.4f}")

## Standalone HTML export (no kernel needed to view; hover + select still work)

In [ ]:
xy.scatter_chart(xy.scatter(x=xs, y=ys, color=depth)).to_html("scatter_chart.html")
print("wrote scatter_chart.html")

## Total restyling — CSS, theme tokens & Tailwind

Every chrome element renders with a stable `data-xy-slot`, and the client's visual
defaults are `:where()` rules (zero specificity) — so **your** CSS, Tailwind
utilities, or `--chart-*` theme tokens always win, no `!important`. The marks are
painted on WebGL (selectors can't reach them), but their props speak CSS: real
`linear-gradient(...)` fills, CSS colors, strokes, dashes.

Each variation restyles the **same** chart, shown via its self-contained
`to_html` export in an iframe so the styling is genuinely applied.

In [ ]:
import html as _html
import pathlib
import re
import tempfile

from IPython.display import HTML, display


def _inject_tailwind(doc):
    # Demo-only: the standalone export ships a strict CSP that blocks CDNs. Relax
    # it and add the Tailwind Play CDN so utilities render here. In a Reflex/
    # Tailwind host, class_names "just work" — no CDN, strict CSP left intact.
    doc = re.sub(
        r'<meta http-equiv="Content-Security-Policy".*?>',
        '<meta http-equiv="Content-Security-Policy" content="default-src \'self\' '
        "'unsafe-inline' 'unsafe-eval' https://cdn.tailwindcss.com data: blob:; img-src * data:;\">",
        doc,
        count=1,
        flags=re.S,
    )
    return doc.replace("</head>", '<script src="https://cdn.tailwindcss.com"></script></head>', 1)


def styled(chart, custom_css=None, tailwind=False, height=420):
    """Render a chart's self-contained HTML (optionally +Tailwind) inside an iframe."""
    with tempfile.NamedTemporaryFile("w+", suffix=".html", delete=True) as f:
        chart.to_html(f.name, custom_css=custom_css)
        doc = pathlib.Path(f.name).read_text()
    if tailwind:
        doc = _inject_tailwind(doc)
    return HTML(
        '<iframe style="width:100%;height:' + str(height) + 'px;border:0;border-radius:12px" '
        'srcdoc="' + _html.escape(doc, quote=True) + '"></iframe>'
    )


# shared data for the styling variations
sx = rng.normal(0, 1, 4000)
sy = sx * 0.5 + rng.normal(0, 0.6, 4000)
sd = sx**2 + sy**2  # continuous -> colormap


def base_scatter(title, cmap):
    return xy.scatter_chart(
        xy.scatter(x=sx, y=sy, color=sd, colormap=cmap, opacity=0.75, size=5),
        xy.x_axis(label="x"),
        xy.y_axis(label="y"),
        title=title,
        width="100%",
        height=360,
    )

In [ ]:
# Variation 1 — pure CSS + --chart-* theme tokens. Same chart, two skins.
neon = """
.xy{
  --chart-bg:#0b1020; --chart-text:#c7d2fe;
  --chart-grid:rgba(129,140,248,.16); --chart-axis:rgba(199,210,254,.45);
  --chart-tooltip-bg:#1e1b4b; --chart-tooltip-text:#e0e7ff;
  font-family:ui-sans-serif,system-ui; border-radius:16px;
}
.xy [data-xy-slot="title"]{ text-transform:uppercase; letter-spacing:.08em; font-weight:700; }
.xy [data-xy-slot="tooltip"]{ border:1px solid rgba(129,140,248,.5); box-shadow:0 0 26px rgba(99,102,241,.5); }
"""
paper = """
.xy{
  --chart-bg:#faf7f0; --chart-text:#44403c;
  --chart-grid:rgba(120,113,108,.14); --chart-axis:rgba(68,64,60,.5);
  --chart-tooltip-bg:#fffdf8; --chart-tooltip-text:#1c1917;
  font-family:Georgia,'Times New Roman',serif; border:1px solid #e7e0d3; border-radius:6px;
}
.xy [data-xy-slot="title"]{ font-style:italic; font-weight:600; }
.xy [data-xy-slot="tooltip"]{ border:1px solid #d6cfbe; box-shadow:0 8px 22px rgba(120,113,108,.28); }
"""
display(styled(base_scatter("midnight neon", "plasma"), custom_css=neon, height=360))
display(styled(base_scatter("warm paper", "magma"), custom_css=paper, height=360))

In [ ]:
# Variation 2 — the marks themselves speak CSS: gradient fills, dashes, strokes.
tt = np.linspace(0, 12, 240)
wave = np.sin(tt) + 0.12 * np.cumsum(rng.normal(0, 0.1, 240))
area = xy.area_chart(
    xy.area(
        x=tt,
        y=wave,
        color="#3b82f6",
        curve="smooth",
        fill="linear-gradient(currentColor, transparent)",
        line_width=2,
    ),
    xy.line(x=tt, y=wave + 2.4, color="#ec4899", dash="dashed", width=2, curve="smooth"),
    xy.x_axis(label="t"),
    xy.y_axis(label="value"),
    title="gradient area + dashed line (mark props)",
    width="100%",
    height=340,
)
bars = xy.bar_chart(
    xy.bar(
        x=["Q1", "Q2", "Q3", "Q4", "Q5", "Q6"],
        y=[5, 9, 4, 7, 6, 8],
        corner_radius=(10, 0),
        stroke="#1e293b",
        stroke_width=1.25,
        fill="linear-gradient(to top, #6366f1, #a5b4fc)",
    ),
    xy.x_axis(label="quarter"),
    xy.y_axis(label="value"),
    title="rounded, stroked, gradient bars",
    width="100%",
    height=340,
)
_css = ".xy{--chart-text:#334155;font-family:ui-sans-serif,system-ui}"
display(styled(area, custom_css=_css, height=320))
display(styled(bars, custom_css=_css, height=320))

In [ ]:
# Variation 3 — Tailwind utilities on chrome slots (class_names) + root class_name.
tw = xy.scatter_chart(
    xy.scatter(x=sx, y=sy, color=sd, colormap="viridis", opacity=0.8, size=5),
    xy.x_axis(label="x"),
    xy.y_axis(label="y"),
    title="Tailwind-styled chrome",
    width="100%",
    height=360,
    class_name="bg-slate-900 rounded-2xl p-3 ring-1 ring-indigo-500/30",
    class_names={
        "title": "text-indigo-300 font-semibold tracking-wide",
        "tooltip": "rounded-xl bg-slate-800/95 text-slate-100 shadow-xl ring-1 ring-indigo-400/40 px-3 py-2",
        "legend": "text-xs text-slate-300",
        "modebar_button": "hover:bg-indigo-500/20 rounded",
        "tick_label": "text-slate-400",
    },
)
# tailwind=True injects the Play CDN for this standalone demo (needs network);
# a Reflex/Tailwind host already has Tailwind, so class_names apply with no CDN.
styled(
    tw,
    tailwind=True,
    custom_css=".xy{--chart-text:#cbd5e1;--chart-grid:rgba(148,163,184,.18);--chart-bg:transparent}",
    height=380,
)

## Reference recreation — stacked gradient area (Desktop vs Mobile)

The polished analytics look: two smooth **plot-space** gradient areas (Desktop
behind, Mobile in front) with minimal chrome — grid dialed down, y-axis labels
hidden, modebar off, and the legend moved to bottom-center. `space:"plot"` is the
key: a per-mark (default) gradient fades each daily column independently and
looks streaky; one plot-space gradient gives the smooth field below.

In [ ]:
# Two smooth plot-space gradient areas + minimal chrome (CSS-only chrome tweaks).
n = 90
days = np.datetime64("2024-04-01") + np.arange(n).astype("timedelta64[D]")
drift = np.linspace(0, 18, n)
desktop = 32 + drift + 40 * rng.random(n)
mobile = desktop * (0.40 + 0.20 * rng.random(n))

analytics = xy.area_chart(
    xy.area(
        x=days,
        y=desktop,
        name="Desktop",
        color="#880808",
        line_width=1.5,
        line_opacity=1,
        fill={"gradient": "linear-gradient(to bottom, #6E260E, #FF0000)", "space": "plot"},
    ),
    xy.x_axis(label=None),
    xy.y_axis(label=None),
    width="100%",
    height=320,
    padding=[18, 20, 46, 20],
)
analytics_css = """
.xy{ --chart-bg:#ffffff; --chart-text:#94a3b8; --chart-grid:rgba(15,23,42,.045);
  --chart-axis:transparent; font-family:ui-sans-serif,system-ui,'Segoe UI',sans-serif; }
.xy [data-xy-slot="modebar"]{ display:none !important; }
.xy [data-xy-slot="tick_label"][data-xy-axis="y"]{ display:none !important; }
.xy [data-xy-slot="tick_label"]{ font-size:13px; color:#94a3b8; }
.xy [data-xy-slot="legend"]{
  top:auto !important; right:auto !important; left:50% !important; bottom:0 !important;
  transform:translateX(-50%); flex-direction:row !important; gap:24px !important;
  max-height:none !important; overflow:visible !important; background:transparent !important;
  box-shadow:none !important; padding:0 !important; }
.xy [data-xy-slot="legend_item"]{ font-size:13px; color:#64748b; }
"""
styled(analytics, custom_css=analytics_css, height=340)